## Routing policy notes (author's summary)

Here is everything I added for the routing policy, in four parts.

Files
routing_policy.py is the decision engine. It takes your zone or ZIP forecast (pred_critical_1h, pred_other_1h) plus live state and does two things.
Repositioning: it chooses where idle units should stage, maximizing expected demand covered within 8 minutes (critical) or 15 minutes (other). It uses a greedy pick, a Hungarian assignment of units to the chosen spots, and a minimum-gain threshold so units don’t shuffle for tiny improvements. It also enforces a one-unit-per-borough floor and weights demand by vulnerability.
Dispatch: critical calls get the fastest unit. For other calls it picks, among near-fastest units, the one whose departure hurts coverage least.
Live inputs: unit status, a traffic multiplier, the fleet’s busy fraction, an optional nowcast correction, and a failsafe that holds positions if the forecast is over 45 minutes old.
Return versus relocation: units coming back from a job get uncapped placement. Discretionary relocation of idle units is capped and rate-limited. I added this split after the first tests showed units stranded at hospitals.
Travel times: a placeholder (straight-line distance × 1.35 at 30 km/h). You can pass a real road-network matrix as tt_base.
`ems_replay.py` connects your model to the policy at ZIP level. It builds the ZIP table (centroids, borough, zone), converts your pred output into per-bucket forecasts, builds a static baseline from training data, loads real incident timestamps, and replays them minute by minute under a static or dynamic policy. It then summarizes response times and per-borough equity. It is called `ems_replay.py` because a pip package named `replay` was shadowing the original name.
demo_policy.py is a synthetic test harness, useful only as a code check.
Notebook cells
Settings: replay day, fleet size, hospital ZIPs.
ZIP table, from the Census file or your own CSV with a column rename, plus ZIP-level vulnerability.
Forecast tables and the static baseline.
Real incidents from your ems_raw CSV.
Replay both policies and compare.

Cells 6 to 8 load a Granite model and turn one cycle’s moves into a short dispatcher briefing. They include a check that falls back to the plain reason strings if the model mentions a ZIP that isn’t in the moves.

What the testing showed
I fixed several bugs: duplicate zone ids in the demo, the stranded-units problem, and an unfair static baseline.
With a fair baseline on synthetic data, dynamic deployment was only marginally better than a layout optimized once for the daily average.
A replay cycle took about 30 ms at 190 ZIPs and 200 units.

None of this says anything about real NYC performance. Your replay on 2022 data is the first meaningful test.

Assumptions to check
The bucket column is the start of the bucket, with the forecast issued 30 minutes later. If not, set issue_offset to zero.
full has y (all calls) and y_critical, and risk.vulnerability returns a Series indexed by zone.
Fleet size, service times, hospital ZIPs, the rush-hour traffic rule and travel times are all placeholders. Response times are modelled, not measured.
The Census gazetteer URL is from memory.
Problems you hit
replay attribute errors: caused by the shadowing pip package, fixed by renaming the file and putting it in src with the other modules.
KeyError: 'GEOID': your CSV doesn’t match the gazetteer layout. Rename your own columns when building the centroid table, or use pgeocode if the file has no coordinates.

**Changes since:** a repositioning unit now stays dispatchable while it drives (before, it was unavailable for the whole drive, which handicapped the dynamic policy); ZIP centroids come from our own zone boundaries (no download); hospitals come from `interim/posts.csv`; the replay compares four policies (static, scheduled, forecast, oracle) over several days; the Granite step skips itself when memory is short.

In [1]:
import os, time
import pandas as pd
from dotenv import load_dotenv
from sodapy import Socrata

load_dotenv()
client = Socrata("data.cityofnewyork.us", os.getenv("SOCRATA_TOKEN"), timeout=120)

os.makedirs("ems_raw", exist_ok=True)

cols = "incident_datetime,zipcode,initial_severity_level_code,valid_incident_rspns_time_indc"
PAGE = 10000   # smaller pages are far less likely to time out into a 503

def get_page(where, offset, retries=8):
    for i in range(retries):
        try:
            return client.get("76xm-jjuj", select=cols, where=where,
                              order="incident_datetime", limit=PAGE, offset=offset)
        except Exception as e:
            wait = min(2 ** i, 60)
            print(f"  offset {offset}: {e} -> retry {i+1}/{retries} in {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"Gave up at offset {offset}")

for year in range(2016, 2023):
    final = f"ems_raw/{year}.csv"
    if os.path.exists(final):
        continue
    part_dir = f"ems_raw/{year}_parts"
    os.makedirs(part_dir, exist_ok=True)
    w = f"incident_datetime between '{year}-01-01T00:00:00' and '{year}-12-31T23:59:59'"

    offset = 0
    while True:
        part = f"{part_dir}/{offset:09d}.csv"
        if os.path.exists(part):                  # page already downloaded
            offset += PAGE
            continue
        batch = get_page(w, offset)
        if not batch:
            break
        pd.DataFrame.from_records(batch).to_csv(part, index=False)
        offset += PAGE
        print(f"{year}: {offset:,} rows requested")

    parts = sorted(os.listdir(part_dir))
    if parts:
        pd.concat((pd.read_csv(f"{part_dir}/{p}", dtype={"zipcode": str}) for p in parts),
                  ignore_index=True).to_csv(final, index=False)
        print(year, "saved")

## Settings
Everything you might change is here. **The defaults fit the LinuxONE VM** (8.4 GB RAM, about 5.6 GB free, 3 CPUs): two years of data peak at about 1.6 GB for the panel stage (measured on the VM) and roughly 3.5 GB for the whole notebook (estimated). All seven years will not fit there and will kill the kernel.
On a machine with more memory, widen `EMS_YEARS` and move `TRAIN_START` earlier. If you change `EMS_YEARS`, change the three dates to match: the test period must be inside the loaded years, and training must have at least a few months.

In [2]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "README.md").exists())
sys.path.insert(0, str(ROOT / "src"))
from grease.forecast import features as F, baseline as B, evaluate as E, zones as Z, export as X
from grease.forecast.lgbm import ForecastModel
from grease import risk

EMS_YEARS = [2021, 2022]                         # SAFE DEFAULT for the LinuxONE VM (8.4 GB RAM, ~5.6 GB free, 3 CPUs). On a bigger machine try list(range(2016, 2023))
TEST_START = pd.Timestamp("2022-01-01")          # F11: test on 2022
VAL_START = pd.Timestamp("2021-10-01")           # last quarter of 2021 = validation set (early stopping)
TRAIN_START = pd.Timestamp("2021-01-01")          # with more years loaded, move this earlier (e.g. 2016-01-01)
EXCLUDE_YEARS = []                               # F10: e.g. [2020] to drop COVID from training
FEATURE_GROUPS = F.DEFAULT_GROUPS                # F5-F7: also try adding "weather_extra", "zone_census", "zone_health_access"
PARAMS = {}                                      # LightGBM overrides, e.g. {"n_estimators": 300, "learning_rate": 0.1} for a quick run
DATA = Path("data")                              # notebooks/data

## Panel and features
One row per **ZIP code x 30-minute bucket**, built by streaming one year's CSV at a time (the ~10 million raw rows are never all in memory). A row is an *issue time* (the end of that bucket): `y`, `y_critical`,
the lags and rolling sums (which include the current bucket) and the time features are all known by then. Critical = initial severity code 1-3 (assumption F9, to be confirmed).
Logic lives in `src/grease/forecast/features.py`, which is unit-tested.

In [3]:
panel, rows_read = F.build_panel_from_csvs([f"ems_raw/{y}.csv" for y in EMS_YEARS])

# completeness check against the API's yearly totals: a short year silently biases everything downstream
API_TOTALS = {2016: 1441211, 2017: 1463446, 2018: 1530922, 2019: 1533251, 2020: 1412701, 2021: 1491454, 2022: 1583531}
check = pd.DataFrame({"rows": {y: rows_read[f"ems_raw/{y}.csv"] for y in EMS_YEARS}, "api_total": {y: API_TOTALS[y] for y in EMS_YEARS}})
check["pct_complete"] = (100 * check.rows / check.api_total).round(1)
print(check)
short = check[check.pct_complete < 99].index.tolist()
if short:
    print(f"WARNING: {short} look incomplete. Delete ems_raw/<year>.csv and ems_raw/<year>_parts, then rerun the fetch cell.")

panel = F.add_history_features(panel)   # lags and rolling sums as of the issue bucket, plus time features
print(panel.shape, panel.bucket.min(), "->", panel.bucket.max(), "| zipcodes:", panel.zipcode.nunique(), f"| ~{panel.memory_usage().sum() / 1e9:.1f} GB")

         rows  api_total  pct_complete
2021  1491454    1491454         100.0
2022  1583531    1583531         100.0
(8269440, 21) 2021-01-01 00:00:00 -> 2022-12-31 23:30:00 | zipcodes: 236 | ~0.6 GB


## Link all other data sources onto it (`full`)
EMS counts are combined with everything fetched and cleaned by **`DarraghC.ipynb`** (run that notebook first; its outputs live in `notebooks/data/`):

| Joined on | From | Columns added |
|---|---|---|
| 30-minute `bucket` | `weather_calendar_30min.csv.gz` | weather and holiday columns, and the `eval_` heat / heavy-rain day flags |
| `zipcode` -> `zone` -> | `interim/zip_to_zone.csv` then `interim/zone_context.csv` | borough, location, population, asthma rate, Census columns, hospital access |

**Only the columns the chosen `FEATURE_GROUPS` need are attached** (plus the two `eval_` flags), and each is built from one small lookup repeated over the grid, so `full` stays lean. Add a group to `FEATURE_GROUPS` and rerun from here to test it.
`eval_*` columns only slice results by heat or heavy-rain days and are never model inputs. Which columns the model uses is an explicit list (decision M2), not "everything in the table".

In [4]:
wx = pd.read_csv(DATA / "weather_calendar_30min.csv.gz", parse_dates=["bucket"]).set_index("bucket")
zip_to_zone = pd.read_csv(DATA / "interim" / "zip_to_zone.csv", dtype=str).set_index("zipcode")["zone"]
zone_ctx = pd.read_csv(DATA / "interim" / "zone_context.csv", dtype={"zone": str}).set_index("zone")

full = F.link_context(panel, wx, zip_to_zone, zone_ctx, FEATURE_GROUPS)
added = [c for c in full.columns if c not in panel.columns]
del panel
zone_known = full["borough"].notna().mean() if "borough" in full else float("nan")
print("full:", full.shape, f"| ~{full.memory_usage().sum() / 1e9:.1f} GB | added columns:", added)
print(f"rows whose ZIP is in a known zone: {zone_known:.1%} (the rest are rare ZIPs, about 0.1% of incidents)")
full.head(3).T

full: (8269440, 35) | ~0.9 GB | added columns: ['temp_c', 'feels_like_c', 'precip_mm', 'rain_3h_mm', 'wind_kmh', 'is_holiday', 'is_day_before_holiday', 'is_day_after_holiday', 'eval_is_heat_day', 'eval_is_heavy_rain_day', 'lon', 'lat', 'pop_est', 'borough']
rows whose ZIP is in a known zone: 88.6% (the rest are rare ZIPs, about 0.1% of incidents)


,0,1,2
zipcode,10000,10000,10000
bucket,2021-01-01 00:00:00,2021-01-01 00:30:00,2021-01-01 01:00:00
y,0,0,0
y_critical,0,0,0
y_other,0,0,0
lag_1,NaN,0.0,0.0
lag_2,NaN,NaN,0.0
lag_3,NaN,NaN,NaN
lag_6,NaN,NaN,NaN
lag_48,NaN,NaN,NaN


## Model: expected demand for the next 30 min / 1 h / 2 h / 3 h (decisions D8-D10)
For every ZIP and every issue time we predict the **expected number of incidents in the next window**, separately for **critical** and **other** calls (8 LightGBM Poisson models: 2 acuities x 4 horizons).
The risk index (further down) is a derived, explainable layer on top, not part of the model. Code: `src/grease/forecast/` and `src/grease/risk.py`.

- **Split by date** (F11): train up to the validation start, validate for early stopping, test on 2022. A gap of 3 hours is dropped before each boundary because a training target looks up to 3 hours ahead.
- **Baselines** (M4), scored on exactly the same rows: historical mean for that ZIP x weekday x time of day; that mean x a citywide recent-surge multiplier; and the same window one week earlier.
- **Memory:** targets are computed one at a time as arrays, the feature matrix is built once as float32, and all 8 models share one binned dataset.

In [5]:
features = F.select_features(full.columns, FEATURE_GROUPS)
print(len(features), "features:", features)

b = full["bucket"]
embargo = pd.Timedelta(minutes=30 * max(F.HORIZONS.values()))     # keep a training target from reaching into the next period
complete = F.complete_window_mask(full)                           # the last 3 h of each ZIP have no full future window
in_train = ((b >= TRAIN_START) & (b < VAL_START - embargo)).to_numpy() & ~b.dt.year.isin(EXCLUDE_YEARS).to_numpy() & complete
in_valid = ((b >= VAL_START) & (b < TEST_START - embargo)).to_numpy() & complete
in_test = (b >= TEST_START).to_numpy()
rows_train, rows_valid, rows_test = (np.flatnonzero(m) for m in (in_train, in_valid, in_test))
for name, r in (("train", rows_train), ("validation", rows_valid), ("test", rows_test)):
    assert len(r), f"{name} period is empty: the loaded EMS_YEARS do not cover it. Adjust TRAIN_START / VAL_START / TEST_START in the settings cell."
    print(f"{name:10s} {len(r):>12,} rows   {b.iloc[r].min()} -> {b.iloc[r].max()}")

31 features: ['zipcode', 'y', 'y_critical', 'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_48', 'lag_336', 'roll_2h', 'roll_6h', 'roll_24h', 'roll_7d', 'roll_28d', 'crit_roll_2h', 'hour', 'dow', 'month', 'is_weekend', 'temp_c', 'feels_like_c', 'precip_mm', 'rain_3h_mm', 'wind_kmh', 'is_holiday', 'is_day_before_holiday', 'is_day_after_holiday', 'lon', 'lat', 'pop_est', 'borough']
train         3,091,128 rows   2021-01-01 00:00:00 -> 2021-09-30 20:30:00
validation    1,040,760 rows   2021-10-01 00:00:00 -> 2021-12-31 20:30:00
test          4,134,720 rows   2022-01-01 00:00:00 -> 2022-12-31 23:30:00


In [6]:
model = ForecastModel(features, PARAMS).fit(full, rows_train, rows_valid, targets=lambda a, h: F.target_array(full, a, h))

  t_critical_30m: 3,091,128 training rows, best iteration 268
  t_critical_1h: 3,091,128 training rows, best iteration 305
  t_critical_2h: 3,091,128 training rows, best iteration 321
  t_critical_3h: 3,091,128 training rows, best iteration 299
  t_other_30m: 3,091,128 training rows, best iteration 413
  t_other_1h: 3,091,128 training rows, best iteration 356
  t_other_2h: 3,091,128 training rows, best iteration 403
  t_other_3h: 3,091,128 training rows, best iteration 346


In [7]:
pred = model.predict(full, rows_test)       # pred_critical_30m ... pred_other_3h, pred_all_30m ... (expected counts, not integers)
test = full.iloc[rows_test]
train_small = full.iloc[rows_train][["zipcode", "dow", "hour", "bucket", "roll_2h", "y"]]   # only what the baselines need
surge = B.surge_multiplier(train_small, test)
results = {}                                # horizon -> actuals and every model's predictions, on the same rows
for h in F.HORIZONS:
    t_all = F.target_array(full, "all", h)
    hist = B.historical_mean(train_small, t_all[rows_train], test)
    results[h] = dict(actual=t_all[rows_test], preds={
        "model": pred[f"pred_all_{h}"].to_numpy(), "hist_mean": hist, "hist_x_surge": hist * surge,
        "last_week": B.last_week(full, t_all)[rows_test]})
print("predicted", f"{len(pred):,}", "test rows for horizons", list(F.HORIZONS))

predicted 4,134,720 test rows for horizons ['30m', '1h', '2h', '3h']


In [8]:
slices = {"all test rows": np.ones(len(test), bool),
          "heat days (daily max >= 32C)": (test.eval_is_heat_day == 1).to_numpy(),
          "heavy-rain days": (test.eval_is_heavy_rain_day == 1).to_numpy(),
          "busiest 20 zipcodes": test.zipcode.isin(E.busiest_zipcodes(train_small, 20)).to_numpy()}
if "borough" in test:
    for bname in sorted(test.borough.dropna().unique()):
        slices[f"borough: {bname}"] = (test.borough == bname).to_numpy()

scores = pd.concat([E.score_slices(r["actual"], r["preds"], slices).assign(horizon=h) for h, r in results.items()], ignore_index=True)
scores["horizon"] = pd.Categorical(scores["horizon"], categories=list(F.HORIZONS), ordered=True)

mae = scores.pivot_table(index=["slice", "horizon"], columns="model", values="MAE", observed=True)
mae["model beats best baseline by (%)"] = (100 * (1 - mae["model"] / mae[["hist_mean", "hist_x_surge", "last_week"]].min(axis=1))).round(1)
print("MAE of the expected count (lower is better). Positive % = the model has lower error than the best baseline.")
display(mae[["model", "hist_mean", "hist_x_surge", "last_week", "model beats best baseline by (%)"]].round(4))
print("Poisson deviance (lower is better):")
display(scores.pivot_table(index=["slice", "horizon"], columns="model", values="deviance", observed=True).round(4))
print("Bias of the model (mean predicted / mean actual; 1.0 = unbiased), per slice and horizon:")
display(scores[scores.model == "model"].pivot_table(index="slice", columns="horizon", values="bias", observed=True).round(3))

MAE of the expected count (lower is better). Positive % = the model has lower error than the best baseline.


model                                  model  hist_mean  hist_x_surge  \
slice                        horizon                                    
all test rows                30m      0.3914     0.3915        0.3963   
                             1h       0.5689     0.5731        0.5784   
                             2h       0.8186     0.8295        0.8350   
                             3h       1.0161     1.0329        1.0379   
borough: BRONX               30m      0.6922     0.6992        0.7071   
                             1h       0.9886     1.0013        1.0106   
                             2h       1.4169     1.4391        1.4499   
                             3h       1.7566     1.7876        1.7985   
borough: BROOKLYN            30m      0.6255     0.6275        0.6336   
                             1h       0.8838     0.8936        0.9008   
                             2h       1.2649     1.2851        1.2908   
                             3h       1.5691     1.5982        1.6000   
borough: MANHATTAN           30m      0.3349     0.3339        0.3374   
                             1h       0.4822     0.4865        0.4902   
                             2h       0.7004     0.7116        0.7155   
                             3h       0.8737     0.8933        0.8960   
borough: QUEENS              30m      0.3538     0.3519        0.3578   
                             1h       0.5376     0.5375        0.5435   
                             2h       0.7722     0.7779        0.7845   
                             3h       0.9541     0.9634        0.9711   
borough: STATEN ISLAND       30m      0.4171     0.4158        0.4220   
                             1h       0.6154     0.6172        0.6246   
                             2h       0.8762     0.8861        0.8951   
                             3h       1.0831     1.0965        1.1064   
busiest 20 zipcodes          30m      0.8825     0.8904        0.8995   
                             1h       1.2545     1.2720        1.2828   
                             2h       1.8030     1.8367        1.8482   
                             3h       2.2374     2.2861        2.2956   
heat days (daily max >= 32C) 30m      0.4108     0.4060        0.4167   
                             1h       0.5952     0.5963        0.6062   
                             2h       0.8579     0.8715        0.8767   
                             3h       1.0663     1.0962        1.0908   
heavy-rain days              30m      0.3893     0.3912        0.3946   
                             1h       0.5672     0.5734        0.5766   
                             2h       0.8178     0.8310        0.8329   
                             3h       1.0166     1.0362        1.0363   

model                                 last_week  \
slice                        horizon              
all test rows                30m         0.4637   
                             1h          0.7266   
                             2h          1.0965   
                             3h          1.3788   
borough: BRONX               30m         0.8820   
                             1h          1.3258   
                             2h          1.9450   
                             3h          2.4202   
borough: BROOKLYN            30m         0.7554   
                             1h          1.1629   
                             2h          1.7266   
                             3h          2.1542   
borough: MANHATTAN           30m         0.3986   
                             1h          0.6243   
                             2h          0.9434   
                             3h          1.1892   
borough: QUEENS              30m         0.3919   
                             1h          0.6421   
                             2h          0.9991   
                             3h          1.2691   
borough: STATEN ISLAND       30m         0.4594   
                             1h          0.7469   
 

Poisson deviance (lower is better):


model                                 hist_mean  hist_x_surge  last_week  \
slice                        horizon                                       
all test rows                30m         0.7275        0.7248     5.3965   
                             1h          0.8545        0.8493     6.5137   
                             2h          0.9451        0.9352     6.2985   
                             3h          0.9858        0.9718     5.6574   
borough: BRONX               30m         1.0710        1.0681     8.9863   
                             1h          1.1489        1.1433     8.6187   
                             2h          1.1707        1.1606     6.1919   
                             3h          1.1828        1.1701     4.8617   
borough: BROOKLYN            30m         1.0465        1.0417     8.5431   
                             1h          1.1795        1.1700     9.5531   
                             2h          1.2329        1.2149     7.9739   
                             3h          1.2450        1.2192     6.2956   
borough: MANHATTAN           30m         0.6396        0.6362     4.6664   
                             1h          0.7571        0.7504     5.5858   
                             2h          0.8521        0.8392     5.2384   
                             3h          0.9036        0.8850     4.5961   
borough: QUEENS              30m         0.7597        0.7578     5.0527   
                             1h          0.9389        0.9350     7.2029   
                             2h          1.0807        1.0735     8.4144   
                             3h          1.1368        1.1269     8.3037   
borough: STATEN ISLAND       30m         0.8374        0.8357     5.9419   
                             1h          1.0282        1.0249     8.3452   
                             2h          1.1610        1.1552     9.2295   
                             3h          1.2050        1.1974     8.6566   
busiest 20 zipcodes          30m         1.2009        1.1960    10.4755   
                             1h          1.2128        1.2037     8.0794   
                             2h          1.1978        1.1807     4.2157   
                             3h          1.2092        1.1860     2.9467   
heat days (daily max >= 32C) 30m         0.7541        0.7455     5.6750   
                             1h          0.8814        0.8638     6.7080   
                             2h          0.9811        0.9455     6.3630   
                             3h          1.0368        0.9831     5.6881   
heavy-rain days              30m         0.7240        0.7201     5.3243   
                             1h          0.8534        0.8458     6.4472   
                             2h          0.9444        0.9303     6.1883   
                             3h          0.9852        0.9664     5.5491   

model                                  model  
slice                        horizon          
all test rows                30m      0.6780  
                             1h       0.8046  
                             2h       0.8872  
                             3h       0.9204  
borough: BRONX               30m      1.0353  
                             1h       1.1126  
                             2h       1.1318  
                             3h       1.1397  
borough: BROOKLYN            30m      1.0109  
                             1h       1.1403  
                             2h       1.1857  
                             3h       1.1907  
borough: MANHATTAN           30m      0.5814  
                             1h       0.6896  
                             2h       0.7642  
                             3h       0.8006  
borough: QUEENS              30m      0.6969  
                             1h       0.8901  
                             2h       1.0385  
                             3h       1.0953  
borough: STATEN ISLAND       30m      0.7902  
                             1

Bias of the model (mean predicted / mean actual; 1.0 = unbiased), per slice and horizon:


horizon,30m,1h,2h,3h
slice,,,,
all test rows,0.961,0.961,0.958,0.958
borough: BRONX,0.968,0.968,0.965,0.965
borough: BROOKLYN,0.960,0.959,0.958,0.956
borough: MANHATTAN,0.953,0.953,0.950,0.949
borough: QUEENS,0.962,0.962,0.960,0.959
borough: STATEN ISLAND,0.970,0.968,0.969,0.968
busiest 20 zipcodes,0.968,0.968,0.965,0.964
heat days (daily max >= 32C),0.961,0.961,0.956,0.954
heavy-rain days,0.948,0.948,0.945,0.946


In [9]:
for h in ("1h", "3h"):
    print(f"Calibration, next {h}, all calls: mean predicted vs mean actual by prediction decile (ratio should be close to 1 in every row)")
    display(E.calibration_table(results[h]["actual"], results[h]["preds"]["model"]).round(4))
imp = model.feature_importance()
print("Feature importance (share of total gain, averaged over the 8 models):")
display(imp.mean(axis=1).sort_values(ascending=False).head(15).round(3))

Calibration, next 1h, all calls: mean predicted vs mean actual by prediction decile (ratio should be close to 1 in every row)


,rows,mean_pred,mean_actual,ratio
bin,,,,
0,413425,0.0010,0.0011,0.9201
1,413425,0.0047,0.0057,0.8228
2,413425,0.0686,0.0730,0.9394
3,413424,0.2533,0.2686,0.9428
4,413425,0.4128,0.4350,0.9489
5,413425,0.5843,0.6044,0.9667
6,413424,0.8169,0.8507,0.9603
7,413425,1.1094,1.1636,0.9534
8,413425,1.5608,1.6203,0.9632


Calibration, next 3h, all calls: mean predicted vs mean actual by prediction decile (ratio should be close to 1 in every row)


,rows,mean_pred,mean_actual,ratio
bin,,,,
0,413331,0.0029,0.0031,0.9405
1,413330,0.0142,0.0170,0.8393
2,413330,0.2080,0.2219,0.9376
3,413331,0.7671,0.8167,0.9393
4,413330,1.2432,1.3107,0.9485
5,413330,1.7583,1.8209,0.9656
6,413331,2.4506,2.5630,0.9562
7,413330,3.3262,3.4949,0.9517
8,413330,4.6596,4.8518,0.9604


Feature importance (share of total gain, averaged over the 8 models):


roll_7d         0.706
roll_24h        0.104
zipcode         0.085
hour            0.073
roll_28d        0.011
roll_6h         0.004
dow             0.003
temp_c          0.003
roll_2h         0.003
feels_like_c    0.001
pop_est         0.001
wind_kmh        0.001
rain_3h_mm      0.001
lag_336         0.001
month           0.001
dtype: float64

In [10]:
t0 = test.groupby("bucket")["y"].sum().idxmax()                       # the busiest half-hour in the test period
rows = test.index[test.bucket == t0]
snap = pred.loc[rows].assign(zipcode=test.loc[rows, "zipcode"].to_numpy(), bucket=t0)
zones_now = Z.aggregate_to_zones(snap, zip_to_zone, ["pred_critical_1h", "pred_other_1h"]).set_index("zone")   # M7: ZIP -> zone, exact sum
zones_now["risk_index"] = risk.risk_index(zones_now.pred_critical_1h, zones_now.pred_other_1h, risk.vulnerability(zone_ctx)).round(1)
zones_now = zones_now.join(zone_ctx[["borough"]]).sort_values("risk_index", ascending=False)
print("Snapshot at", t0, "- expected incidents in the next hour per zone, and the 0-100 risk index (D9)")
zones_now.drop(columns="bucket").head(10).round(2)

Snapshot at 2022-03-30 11:00:00 - expected incidents in the next hour per zone, and the 0-100 risk index (D9)


/tmp/ipykernel_8206/960993613.py:8: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  zones_now.head(10).round(2)


,bucket,pred_critical_1h,pred_other_1h,risk_index,borough
zone,,,,,
10456,2022-03-30 11:00:00,1.54,2.51,100.0,BRONX
11212,2022-03-30 11:00:00,1.39,2.31,99.4,BROOKLYN
10457,2022-03-30 11:00:00,1.25,2.32,98.9,BRONX
10029,2022-03-30 11:00:00,1.32,2.00,98.3,MANHATTAN
10467,2022-03-30 11:00:00,1.31,2.00,97.7,BRONX
11207,2022-03-30 11:00:00,1.19,2.28,97.2,BROOKLYN
10451,2022-03-30 11:00:00,1.11,1.93,96.6,BRONX
10453,2022-03-30 11:00:00,1.16,1.79,96.0,BRONX
10035,2022-03-30 11:00:00,1.12,2.04,95.5,MANHATTAN


## Hand-off table for the optimizer and simulator
One row per **zone x 30-minute bucket** of the test period: the model's expected incidents (`pred_*`), the scheduled-policy demand (`hist_all_*`), what actually happened (`actual_*`, for the oracle policy and scoring) and the risk index.
Columns and how to use them are described in `docs/forecast_handoff.md`. Saved next to the models (git-ignored, regenerate by rerunning this notebook).

In [11]:
actual = {(a, h): F.target_array(full, a, h)[rows_test] for a in F.ACUITY for h in F.HORIZONS}
baseline = {h: results[h]["preds"]["hist_mean"] for h in F.HORIZONS}
zone_table = X.zone_forecast_table(test, pred, actual, baseline, zip_to_zone, zone_ctx)
out = DATA / "processed"
out.mkdir(parents=True, exist_ok=True)
zone_table.to_csv(out / "forecast_zone_test.csv.gz", index=False, float_format="%.3f", compression={"method": "gzip", "compresslevel": 1})
print("saved", out / "forecast_zone_test.csv.gz", "|", f"{len(zone_table):,} rows,", zone_table.zone.nunique(), "zones,", len(zone_table.columns), "columns")
zone_table.head(3).T

saved data/processed/forecast_zone_test.csv.gz | 3,101,040 rows, 177 zones, 24 columns


,0,1,2
zone,10001,10001,10001
bucket,2022-01-01 00:00:00,2022-01-01 00:30:00,2022-01-01 01:00:00
pred_critical_30m,0.309193,0.306202,0.295774
actual_critical_30m,1.0,0.0,1.0
pred_other_30m,0.93848,0.924517,0.886287
actual_other_30m,1.0,2.0,3.0
hist_all_30m,1.307692,1.51282,1.128205
pred_critical_1h,0.66203,0.656579,0.599595
actual_critical_1h,1.0,1.0,3.0
pred_other_1h,1.812963,1.732322,1.697556


In [12]:
out = DATA / "processed"
out.mkdir(parents=True, exist_ok=True)
model.save(out / "forecast_models")             # LightGBM text files + meta.json: portable to the VM
scores.to_csv(out / "forecast_scores.csv", index=False)
print("saved models and scores to", out)

saved models and scores to data/processed


## Routing: where should idle ambulances wait, and what does it do to response times?
Replays real 2022 incidents minute by minute under four ways of positioning the same fleet, using **the same incidents, fleet, travel times and service times** for each:

| Policy | Idle units are positioned using... |
|---|---|
| **static** | one layout, optimised once on the average training-period demand, never changed |
| **dynamic: scheduled** | the historical mean demand for that ZIP, weekday and time of day (no ML) |
| **dynamic: forecast** | our model's forecast |
| **dynamic: oracle** | the demand that actually happened in the next hour (impossible in practice: the upper bound) |

How far *forecast* sits above *scheduled*, and how far below *oracle*, tells us whether forecast accuracy matters for dispatch. **Response times are modelled, not measured**: fleet size, scene and hospital times, traffic and travel times (straight line x 1.35 at 30 km/h) are placeholders, so read the differences between policies, not the absolute minutes. **The fleet size matters enormously**: the replay flips from overloaded to comfortable between about 300 and 400 units, which is why the sensitivity table below is part of the result.

In [ ]:
import sys; sys.path.insert(0, ".")          # this folder holds routing_policy.py and ems_replay.py
import ems_replay as R
from routing_policy import PolicyConfig

REPLAY_DAYS = ["2022-07-19", "2022-10-12"]   # each must start on a :00 or :30 boundary and be inside the test year; the first is a heat-wave day
N_UNITS = 330                                # fleet size: an ASSUMPTION (no public fleet data), calibrated so the STATIC layout lands near real NYC performance
SENSITIVITY_UNITS = [300, 370]               # also replayed on the first day, because results swing a lot near the tipping point
# Why 330: about 4,600 calls a day x about an hour per job = about 190 busy units on average. With 200-300 units the replay is overloaded (calls queue for
# hours, 20% unserved); at 400+ the static layout already reaches ~88% of critical calls in 8 min, better than reality (roughly 60-65%). 330 gives ~71%.

In [ ]:
heat = test.loc[test.eval_is_heat_day == 1, "bucket"].dt.normalize().value_counts().sort_index()
print("heat days in the test year (pick some for REPLAY_DAYS):", [str(d.date()) for d in heat.index][:12])

In [ ]:
# ZIP table: every ZIP gets the centroid of its zone (our own boundaries, so no download is needed)
cent = pd.DataFrame({"zone": zip_to_zone}).join(zone_ctx[["lat", "lon"]], on="zone")[["lat", "lon"]].dropna()
zips = R.build_zip_table(test["zipcode"].unique(), cent, zip_to_zone, zone_ctx)
zip_vuln = zips["zone"].map(risk.vulnerability(zone_ctx))

# real hospital ZIPs: after a call the unit drives to the nearest of these
posts = pd.read_csv(DATA / "interim" / "posts.csv", dtype=str)
HOSPITAL_ZIPS = sorted(set(posts.loc[posts.post_type == "hospital", "zipcode"].dropna()) & set(zips.index))
print(zips.shape, "ZIPs |", len(HOSPITAL_ZIPS), "hospital ZIPs")
zips.head()

In [ ]:
# demand sources for the three dynamic policies (all ZIP level, next hour, same columns as the model's forecast)
crit_1h, oth_1h = F.target_array(full, "critical", "1h"), F.target_array(full, "other", "1h")
sources = {
    "dynamic: scheduled": pd.DataFrame({"pred_critical_1h": B.historical_mean(train_small, crit_1h[rows_train], test),
                                        "pred_other_1h": B.historical_mean(train_small, oth_1h[rows_train], test)}, index=test.index),
    "dynamic: forecast": pred[["pred_critical_1h", "pred_other_1h"]],
    "dynamic: oracle": pd.DataFrame({"pred_critical_1h": crit_1h[rows_test], "pred_other_1h": oth_1h[rows_test]}, index=test.index),
}
static_fc = R.static_forecast(full[["zipcode", "y", "y_critical"]].iloc[rows_train])      # training period only: the static layout must not see 2022
print({k: v.shape for k, v in sources.items()}, "| static layout from", len(static_fc), "ZIPs")

In [ ]:
all_days, runs_by_day = [], {}
for day in REPLAY_DAYS:
    start = pd.Timestamp(day); end = start + pd.Timedelta("1D")
    incidents = R.load_incidents(["ems_raw/2022.csv"], start, end)
    kw = dict(n_units=N_UNITS, start=start, end=end, hospital_zips=HOSPITAL_ZIPS)
    runs = [R.run_replay(zips, {}, incidents, static_fc, zip_vuln, dynamic=False, label="static", **kw)]
    for name, src in sources.items():
        runs.append(R.run_replay(zips, R.forecast_by_bucket(src, test, start, end), incidents, static_fc, zip_vuln,
                                 dynamic=True, label=name, **kw))
    overall, by_borough = R.summarize(runs, zips)
    print(f"\n=== {start.date()}: {len(incidents):,} incidents, {N_UNITS} units ===")
    display(overall)
    print("% of critical calls reached within 8 minutes, by borough:")
    display(by_borough)
    all_days.append(overall.assign(day=str(start.date())))
    runs_by_day[day] = runs
if len(REPLAY_DAYS) > 1:
    print("\nAverage over the replay days:")
    display(pd.concat(all_days).groupby(level=0)[["crit_within_target_pct", "crit_mean_min", "crit_p90_min", "all_mean_min", "moves"]].mean().round(2).reindex(
        ["static", "dynamic: scheduled", "dynamic: forecast", "dynamic: oracle"]))

In [ ]:
# fleet-size sensitivity on the first replay day: how much do the conclusions depend on the fleet assumption?
day = pd.Timestamp(REPLAY_DAYS[0]); end = day + pd.Timedelta("1D")
incidents = R.load_incidents(["ems_raw/2022.csv"], day, end)
tab = {}
for n in sorted({*SENSITIVITY_UNITS, N_UNITS}):
    kw = dict(n_units=n, start=day, end=end, hospital_zips=HOSPITAL_ZIPS)
    runs = [R.run_replay(zips, {}, incidents, static_fc, zip_vuln, dynamic=False, label="static", **kw)]
    for name, src in sources.items():
        runs.append(R.run_replay(zips, R.forecast_by_bucket(src, test, day, end), incidents, static_fc, zip_vuln, dynamic=True, label=name, **kw))
    o, _ = R.summarize(runs, zips)
    tab[n] = o[["crit_within_target_pct", "crit_mean_min", "crit_p90_min", "unserved"]]
sens_table = pd.concat(tab, names=["units"]).unstack("policy").round(1)
print(f"{day.date()}: critical calls reached within 8 min (%), mean and 90th-percentile minutes, and calls still unserved, by fleet size")
display(sens_table["crit_within_target_pct"][["static", "dynamic: scheduled", "dynamic: forecast", "dynamic: oracle"]])
display(sens_table["crit_p90_min"][["static", "dynamic: scheduled", "dynamic: forecast", "dynamic: oracle"]])
display(sens_table["unserved"][["static", "dynamic: forecast"]])

### Dispatcher briefing (generative AI, IBM Granite)
Turns one repositioning cycle's moves into a short plain-language briefing, with a guard: if the model mentions a ZIP that is not in the moves, the briefing falls back to the plain reason strings.
The Granite 2B model needs about 5 GB of free memory plus a ~5 GB download, so this cell **skips itself when less than 8 GB is available** (the LinuxONE VM has about 5.6 GB free) and the briefing then shows the plain reasons.
To run it on the VM, use a quantised build through llama.cpp (installed in the VM image), or run this section in a fresh kernel after saving the replay results.

In [ ]:
import os, re

def free_gb():
    try:
        import psutil
        return psutil.virtual_memory().available / 1e9
    except ImportError:
        for line in open("/proc/meminfo") if os.path.exists("/proc/meminfo") else []:
            if line.startswith("MemAvailable"):
                return int(line.split()[1]) / 1e6
        return float("inf")                      # unknown: assume enough

RUN_BRIEFING = True
MODEL = "ibm-granite/granite-3.3-2b-instruct"      # check Hugging Face for the current name
tok = llm = None
if RUN_BRIEFING and free_gb() >= 8:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(MODEL)
    llm = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True).eval()
else:
    print(f"Skipping the Granite model ({free_gb():.1f} GB available, 8 needed): briefings will show the plain reason strings.")

In [ ]:
dyn = runs_by_day[REPLAY_DAYS[0]][2]                # "dynamic: forecast" on the first replay day
cy = dyn["cycles"]
when = cy.loc[cy["moves"].idxmax(), "time"]         # the busiest cycle; or any timestamp you like

def cycle_facts(when, top=5):
    c = dyn["cycles"].set_index("time").loc[when]
    mv = dyn["moves"][dyn["moves"].time == when].sort_values("gain", ascending=False).head(top)
    return c, mv

def build_prompt(c, mv, when):
    return (
        "You are briefing an EMS dispatcher. Use ONLY the facts below. Do not add numbers, "
        "locations, causes or advice that are not listed. Write at most 4 short plain-language "
        "sentences, most important move first.\n\n"
        f"Time: {when:%H:%M}\n"
        f"Moves recommended this cycle: {int(c['moves'])} (top {len(mv)} shown)\n"
        f"Expected critical demand within 8 minutes of a unit: "
        f"{c['crit_covered_before']:.0%} before, {c['crit_covered_after']:.0%} after\n"
        "Moves:\n" + "\n".join(f"- {r}" for r in mv["reason"])
    )

def briefing(when):
    c, mv = cycle_facts(when)
    plain = "\n".join(mv["reason"])
    if llm is None:
        return plain
    import torch
    msgs = [{"role": "user", "content": build_prompt(c, mv, when)}]
    inputs = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=150, do_sample=False)
    text = tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    allowed = set(mv.from_zip) | set(mv.to_zip)            # guard: any ZIP the model mentions must come from the moves
    return plain if set(re.findall(r"\b\d{5}\b", text)) - allowed else text

print(f"Cycle at {when}:")
print(briefing(when))